In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
from pathlib import Path
import kagglehub

# Baixar o dataset
pasta = Path(kagglehub.dataset_download("saviovianna/queimadas-inpe"))
arquivos = sorted(pasta.glob("qi_??_2024.csv"))

# ATUALIZAÇÃO: Adiciona aqui os nomes exatos das novas colunas que queres extrair
colunas = [
    "lat", "lon", "data_hora_gmt", "municipio", "estado",
    "risco_fogo", "bioma", "temperatura", "ponto_orvalho_2m",
    "precipitacao", "vento_velocidade" # Exemplo de novas métricas
]

partes = []
for arquivo in arquivos:
    parte = pd.read_csv(arquivo, usecols=colunas)
    partes.append(parte)

df = pd.concat(partes, ignore_index=True)
df["data_hora_gmt"] = pd.to_datetime(df["data_hora_gmt"], errors="coerce")

100%|██████████| 168M/168M [00:03<00:00, 53.1MB/s]

Extracting files...


qi_01_2024.csv 88120 registros
qi_02_2024.csv 73315 registros
qi_03_2024.csv 60176 registros
qi_04_2024.csv 27477 registros
qi_05_2024.csv 75364 registros
qi_06_2024.csv 153672 registros
qi_07_2024.csv 225374 registros
qi_08_2024.csv 596641 registros
qi_09_2024.csv 1081937 registros
qi_10_2024.csv 479794 registros
qi_11_2024.csv 258626 registros
qi_12_2024.csv 127537 registros
Total de 2024: 3248033 registros
Período: 2024-01-01 00:00:00 a 2024-12-31 23:00:00


,lat,lon,data_hora_gmt,municipio,estado,risco_fogo,bioma,temperatura,ponto_orvalho_2m
0,-12.5667,-41.4364,2024-01-01 00:00:00,LENÇÓIS,BAHIA,0.70,Caatinga,26.0,18.0
1,-12.5662,-41.4611,2024-01-01 00:00:00,LENÇÓIS,BAHIA,0.83,Caatinga,26.0,18.0
2,-18.0629,-57.3721,2024-01-01 00:00:00,CORUMBÁ,MATO GROSSO DO SUL,0.00,Pantanal,30.0,23.6
3,-18.0625,-57.3927,2024-01-01 00:00:00,CORUMBÁ,MATO GROSSO DO SUL,0.01,Pantanal,29.8,24.1
4,-18.0629,-57.3721,2024-01-01 01:00:00,CORUMBÁ,MATO GROSSO DO SUL,0.00,Pantanal,29.6,24.0


In [ ]:
df.columns

Index(['lat', 'lon', 'data_hora_gmt', 'municipio', 'estado', 'risco_fogo',
       'bioma', 'temperatura', 'ponto_orvalho_2m'],
      dtype='object')

In [ ]:
import numpy as np

# Temperatura do ar e ponto de orvalho em graus Celsius.
t = pd.to_numeric(df["temperatura"], errors="coerce")
td = pd.to_numeric(df["ponto_orvalho_2m"], errors="coerce")
validos = t.notna() & td.notna() & (td <= t)

# Fórmula de Magnus: umidade relativa do ar em porcentagem.
umidade_calculada = 100 * np.exp(
    (17.625 * td / (243.04 + td))
    - (17.625 * t / (243.04 + t))
)
df["umidade_ar_pct"] = umidade_calculada.where(validos).clip(0, 100)

print("Linhas:", len(df))
print("Umidade calculada:", df["umidade_ar_pct"].notna().sum())
print("Temperatura ou ponto de orvalho ausente:", (t.isna() | td.isna()).sum())
print("Ponto de orvalho acima da temperatura:", (t.notna() & td.notna() & (td > t)).sum())
display(df[["temperatura", "ponto_orvalho_2m", "umidade_ar_pct"]].head(10).round(1))

Linhas: 3248033
Umidade calculada: 2938392
Temperatura ou ponto de orvalho ausente: 309641
Ponto de orvalho acima da temperatura: 0


,temperatura,ponto_orvalho_2m,umidade_ar_pct
0,26.0,18.0,61.4
1,26.0,18.0,61.4
2,30.0,23.6,68.6
3,29.8,24.1,71.5
4,29.6,24.0,71.9
5,29.5,24.1,72.8
6,26.8,17.1,55.3
7,24.1,20.5,80.3
8,26.4,19.1,64.2
9,24.1,22.2,89.1


In [ ]:
# Os 12 CSVs reunidos cobrem todo o ano de 2024.
print("Período:", df["data_hora_gmt"].min(), "a", df["data_hora_gmt"].max())
print("Registros no arquivo:", len(df))

# Contagem pelo nome do município informado no dataset.
sao_joao = df[df["municipio"].astype(str).str.contains("JOÃO DA BOA VISTA", case=False, na=False)]
print("Registros em São João da Boa Vista:", len(sao_joao))

# Distância até o ponto central da Serra usado no notebook anterior.
lat_centro, lon_centro = -21.947818246740198, -46.78653641223045
lat = np.radians(pd.to_numeric(df["lat"], errors="coerce"))
lon = np.radians(pd.to_numeric(df["lon"], errors="coerce"))
lat0, lon0 = np.radians(lat_centro), np.radians(lon_centro)
a = np.sin((lat - lat0) / 2)**2 + np.cos(lat0) * np.cos(lat) * np.sin((lon - lon0) / 2)**2
distancia_km = 6371 * 2 * np.arcsin(np.sqrt(a.clip(0, 1)))

perto_5km = df[distancia_km <= 5]
perto_10km = df[distancia_km <= 10]
print("Registros até 5 km do ponto central:", len(perto_5km))
print("Registros até 10 km do ponto central:", len(perto_10km))
display(perto_10km[["data_hora_gmt", "municipio", "lat", "lon", "temperatura", "umidade_ar_pct"]].head(10))

Período: 2024-01-01 00:00:00 a 2024-12-31 23:00:00
Registros no arquivo: 3248033
Registros em São João da Boa Vista: 103
Registros até 5 km do ponto central: 20
Registros até 10 km do ponto central: 100


,data_hora_gmt,municipio,lat,lon,temperatura,umidade_ar_pct
230862,2024-04-11 17:00:00,SÃO JOÃO DA BOA VISTA,-21.99773,-46.802350,29.1,54.175967
239536,2024-04-23 17:00:00,SÃO JOÃO DA BOA VISTA,-22.01036,-46.774740,28.9,31.041293
262217,2024-05-07 16:00:00,SÃO JOÃO DA BOA VISTA,-21.98989,-46.768415,28.9,47.115064
293667,2024-05-19 16:00:00,ÁGUAS DA PRATA,-21.96388,-46.706170,27.2,44.595802
296619,2024-05-21 04:00:00,SÃO JOÃO DA BOA VISTA,-22.01409,-46.829400,17.2,84.695400
349154,2024-06-08 04:00:00,SÃO JOÃO DA BOA VISTA,-21.98126,-46.824680,12.6,90.583328
349155,2024-06-08 04:00:00,SÃO JOÃO DA BOA VISTA,-21.98429,-46.826660,12.6,90.583328
363100,2024-06-11 16:00:00,SÃO JOÃO DA BOA VISTA,-21.98455,-46.826030,26.5,35.715132
381784,2024-06-14 16:00:00,SÃO JOÃO DA BOA VISTA,-21.98741,-46.762260,25.8,39.002215
382658,2024-06-14 17:00:00,SÃO JOÃO DA BOA VISTA,-21.98783,-46.758110,26.4,36.653291


In [ ]:
# Focos registrados em Cerrado ou Mata Atlântica (dados de todo o ano de 2024).
df_biomas = df[df["bioma"].astype(str).str.contains("CERRADO|MATA ATL", case=False, na=False)].copy()

print("Registros dos dois biomas:", len(df_biomas))
print("Por bioma:")
print(df_biomas["bioma"].value_counts())
print("Desses, no estado de São Paulo:", len(df_biomas[df_biomas["estado"] == "SÃO PAULO"]))
display(df_biomas[["data_hora_gmt", "bioma", "estado", "municipio", "temperatura", "umidade_ar_pct"]].head())

Registros dos dois biomas: 1152595
Por bioma:
bioma
Cerrado           944155
Mata Atlântica    208440
Name: count, dtype: int64
Desses, no estado de São Paulo: 73467


,data_hora_gmt,bioma,estado,municipio,temperatura,umidade_ar_pct
10,2024-01-01 03:00:00,Mata Atlântica,PERNAMBUCO,SIRINHAÉM,26.6,82.629974
11,2024-01-01 04:00:00,Mata Atlântica,BAHIA,SANTA CRUZ CABRÁLIA,24.4,89.714979
12,2024-01-01 04:00:00,Mata Atlântica,BAHIA,SANTA CRUZ CABRÁLIA,24.4,89.714979
13,2024-01-01 04:00:00,Mata Atlântica,BAHIA,CANAVIEIRAS,25.2,87.629167
14,2024-01-01 04:00:00,Mata Atlântica,BAHIA,SANTA LUZIA,23.8,89.671004


In [ ]:
# Alvo y: índice de risco de fogo já calculado no dataset (escala de 0 a 1).
risco = pd.to_numeric(df_biomas["risco_fogo"], errors="coerce")
dados_risco = df_biomas.loc[risco.between(0, 1)].copy()
dados_risco["y_risco_fogo"] = risco.loc[dados_risco.index]

print("Registros dos biomas:", len(df_biomas))
print("Risco válido (0 a 1):", len(dados_risco))
print("Risco ausente ou inválido:", len(df_biomas) - len(dados_risco))
print("Resumo do y:")
print(dados_risco["y_risco_fogo"].describe().round(3))
display(dados_risco[["bioma", "temperatura", "umidade_ar_pct", "y_risco_fogo"]].head(10))

Registros dos biomas: 1152595
Risco válido (0 a 1): 1152595
Risco ausente ou inválido: 0
Resumo do y:
count    1152595.000
mean           0.863
std            0.294
min            0.000
25%            0.970
50%            1.000
75%            1.000
max            1.000
Name: y_risco_fogo, dtype: float64


,bioma,temperatura,umidade_ar_pct,y_risco_fogo
10,Mata Atlântica,26.6,82.629974,1.0000
11,Mata Atlântica,24.4,89.714979,0.4125
12,Mata Atlântica,24.4,89.714979,0.5900
13,Mata Atlântica,25.2,87.629167,0.3400
14,Mata Atlântica,23.8,89.671004,0.7400
15,Mata Atlântica,24.7,82.910121,0.4500
16,Cerrado,22.7,89.040268,0.0000
17,Mata Atlântica,25.1,86.041024,0.1600
18,Mata Atlântica,23.5,90.198297,0.1700
29,Mata Atlântica,25.1,85.520104,0.2700


In [ ]:
# Mantém somente linhas que o Arduino poderá representar.
dados_risco = dados_risco.dropna(
    subset=["temperatura", "umidade_ar_pct", "data_hora_gmt"]
).copy()

# O mês vem da data; não exige outro sensor.
dados_risco["mes"] = dados_risco["data_hora_gmt"].dt.month
dados_risco["mes_sen"] = np.sin(2 * np.pi * dados_risco["mes"] / 12)
dados_risco["mes_cos"] = np.cos(2 * np.pi * dados_risco["mes"] / 12)

X = dados_risco[["temperatura", "umidade_ar_pct", "mes_sen", "mes_cos"]].copy()
X["bioma_cerrado"] = (dados_risco["bioma"].str.upper() == "CERRADO").astype(int)
print("Registros prontos para modelar:", len(X))

Registros prontos para modelar: 1016697


In [ ]:
y = dados_risco['y_risco_fogo']

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.25)

# Treinanando modelos de regressão : DecisionTree, Mlp, GradientBoost

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score



In [ ]:
arvore_modelo = DecisionTreeRegressor(max_depth=8, min_samples_leaf=200, random_state=42)

arvore_modelo.fit(X_train, y_train)
pred_arvore = arvore_modelo.predict(X_test)
print("MAE", mean_absolute_error(y_test, pred_arvore))
print("R²: ", r2_score(y_test, pred_arvore) )


MAE 0.11751818232638851
R²:  0.5247807863791776


In [ ]:
gradient_Boost = HistGradientBoostingRegressor(max_iter=100, max_leaf_nodes=15, min_samples_leaf=100, random_state=42)

gradient_Boost.fit(X_train, y_train)
pred_gradient_Boost = gradient_Boost.predict(X_test)
print("MAE", mean_absolute_error(y_test, pred_gradient_Boost))
print("R²: ", r2_score(y_test, pred_gradient_Boost) )

MAE 0.11432531639248632
R²:  0.5562540977867834


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 1. Criar o Pipeline juntando a padronização e a Rede Neural
pipeline_mlp = Pipeline([
    ('escalonador', StandardScaler()),
    ('modelo', MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        max_iter=1000,
        random_state=42,
        early_stopping=True
    ))
])

# 2. Treinar o modelo com os seus dados de treino
pipeline_mlp.fit(X_train, y_train)

# 3. Fazer as previsões no conjunto de teste
previsoes_mlp = pipeline_mlp.predict(X_test)

# 4. Avaliar o desempenho
mse_mlp = mean_squared_error(y_test, previsoes_mlp)
mae_mlp = mean_absolute_error(y_test, previsoes_mlp)
r2_mlp = r2_score(y_test, previsoes_mlp)

print(f"Erro Quadrático Médio (MSE) - MLP: {mse_mlp:.4f}")
print(f"Erro Absoluto Médio (MAE) - MLP: {mae_mlp:.4f}")
print(f"R² (Coeficiente de Determinação) - MLP: {r2_mlp:.4f}")

Erro Quadrático Médio (MSE) - MLP: 0.0393
Erro Absoluto Médio (MAE) - MLP: 0.1092
R² (Coeficiente de Determinação) - MLP: 0.5773


# Aplicando tecnicas de melhorias

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

# Espaço de parâmetros para a Árvore
param_arvore = {
    'max_depth': [3, 5, 8, 10, 15, None],
    'min_samples_leaf': [10, 50, 100, 200],
    'min_samples_split': [10, 50, 100, 200]
}

# Configuração da busca aleatória
random_arvore = RandomizedSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_distributions=param_arvore,
    n_iter=10,  #
    cv=3,
    scoring='neg_mean_absolute_error',
    random_state=42
)

# Treinamento
random_arvore.fit(X_train, y_train)
melhor_arvore = random_arvore.best_estimator_

# Previsão e Avaliação
pred_arvore = melhor_arvore.predict(X_test)
print("--- Árvore de Decisão Otimizada ---")
print("Melhores parâmetros:", random_arvore.best_params_)
print("MAE:", mean_absolute_error(y_test, pred_arvore))
print("R²: ", r2_score(y_test, pred_arvore))

--- Árvore de Decisão Otimizada ---
Melhores parâmetros: {'min_samples_split': 10, 'min_samples_leaf': 10, 'max_depth': None}
MAE: 0.09378092700588758
R²:  0.6066840361954964


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

# Espaço de parâmetros para o Gradient Boosting
param_gb = {
    'max_iter': [50, 100, 150, 200],
    'max_leaf_nodes': [10, 15, 30, 50],
    'min_samples_leaf': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2]
}

# Configuração da busca aleatória
random_gb = RandomizedSearchCV(
    HistGradientBoostingRegressor(random_state=42),
    param_distributions=param_gb,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    random_state=42
)

# Treinamento
random_gb.fit(X_train, y_train)
melhor_gb = random_gb.best_estimator_

# Previsão e Avaliação
pred_gb = melhor_gb.predict(X_test)
print("--- Gradient Boosting Otimizado ---")
print("Melhores parâmetros:", random_gb.best_params_)
print("MAE:", mean_absolute_error(y_test, pred_gb))
print("R²: ", r2_score(y_test, pred_gb))

--- Gradient Boosting Otimizado ---
Melhores parâmetros: {'min_samples_leaf': 50, 'max_leaf_nodes': 50, 'max_iter': 150, 'learning_rate': 0.2}
MAE: 0.10635364088200426
R²:  0.5876159182149123


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

# Espaço de parâmetros para a MLP (precisa usar o prefixo 'modelo__' por causa do Pipeline)
param_mlp = {
    'modelo__hidden_layer_sizes': [(32,), (64, 32), (100, 50)],
    'modelo__activation': ['relu', 'tanh'],
    'modelo__alpha': [0.0001, 0.001, 0.01],
    'modelo__learning_rate_init': [0.001, 0.01]
}

# Pipeline obrigatório para a MLP
pipeline_mlp = Pipeline([
    ('escalonador', StandardScaler()),
    ('modelo', MLPRegressor(max_iter=1000, random_state=42, early_stopping=True))
])

# Configuração da busca aleatória
random_mlp = RandomizedSearchCV(
    pipeline_mlp,
    param_distributions=param_mlp,
    n_iter=5,   # Reduzido para 5 porque treinar redes neurais demora um pouco mais
    cv=3,       # Validação cruzada
    scoring='neg_mean_absolute_error',
    random_state=42
)

# Treinamento
random_mlp.fit(X_train, y_train)
melhor_mlp = random_mlp.best_estimator_

# Previsão e Avaliação
pred_mlp = melhor_mlp.predict(X_test)
print("--- Rede Neural (MLP) Otimizada ---")
print("Melhores parâmetros:", random_mlp.best_params_)
print("MAE:", mean_absolute_error(y_test, pred_mlp))
print("R²: ", r2_score(y_test, pred_mlp))

--- Rede Neural (MLP) Otimizada ---
Melhores parâmetros: {'modelo__learning_rate_init': 0.001, 'modelo__hidden_layer_sizes': (100, 50), 'modelo__alpha': 0.01, 'modelo__activation': 'relu'}
MAE: 0.11162366500344338
R²:  0.5694369818684393


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Função para transformar o risco contínuo (0 a 1) em categorias (0, 1, 2, 3)
def categorizar_risco(risco):
    if risco < 0.35:
        return 0  # Baixo
    elif risco < 0.70:
        return 1  # Médio
    elif risco < 0.90:
        return 2  # Alto
    else:
        return 3  # Crítico

# Aplicar a classificação no teu y_train e y_test
y_train_class = y_train.apply(categorizar_risco)
y_test_class = y_test.apply(categorizar_risco)

# 2. Criar e treinar o Classificador de Árvore de Decisão
classificador = DecisionTreeClassifier(max_depth=6, random_state=42)
classificador.fit(X_train, y_train_class)

# 3. Fazer as previsões
pred_class = classificador.predict(X_test)

# 4. Avaliar o desempenho (Acurácia e Relatório)
acuracia = accuracy_score(y_test_class, pred_class)

print("--- Classificador de Risco de Fogo ---")
print(f"Acurácia (Taxa de Acerto): {acuracia * 100:.2f}%")
print("\nRelatório Detalhado:")
print(classification_report(y_test_class, pred_class))

--- Classificador de Risco de Fogo ---
Acurácia (Taxa de Acerto): 82.24%

Relatório Detalhado:
              precision    recall  f1-score   support

           0       0.62      0.58      0.59     31042
           1       0.33      0.06      0.10     14750
           2       0.00      0.00      0.00     12274
           3       0.86      0.97      0.91    196109

    accuracy                           0.82    254175
   macro avg       0.45      0.40      0.40    254175
weighted avg       0.75      0.82      0.78    254175



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
